# Repository note

This notebook is an output-pruned archival copy of the executed Kaggle notebook. Code, execution metadata, and textual outputs are retained; embedded image display payloads were removed to keep the Git repository compact. SHA256 of the original executed Kaggle notebook: `f8d3864da15528275bd834bce3ce8fe8ac055900a2309f33f1947fc8b0d8bf15`.


# 1. FreeFine Final Geometry — Qualitative Comparison (NO GPU)

Creates presentation-ready visual comparisons from the already-generated balanced-200 outputs.

### Outputs
- Resize severe: Source | Coarse | FreeFine baseline | SGR-EPSREC | SGR-MIDHF+EPSREC
- Resize non-severe: Source | Coarse | FreeFine baseline | SGR-RING8
- Move: Source | Coarse | FreeFine baseline | SGR-RING4
- Rotate: Source | Coarse | FreeFine baseline (both final proposals intentionally keep baseline here)
- Full-frame and target-region zoom grids
- `selected_cases_manifest.csv`
- ZIP with all generated figures

### Required Kaggle inputs
Use the same inputs as Step 1:
- `freefine-sample-metadata`
- `geobench2d-coarse-img`
- `geobench2d-metrics-subset`
- `freefine-geobench2d-bggen`
- notebook output `freefine-final-exhaustive-run-account-b-guidan`

**Accelerator: None. Internet: OFF is fine.**


In [1]:
import os,glob,json,csv,random,hashlib,math,shutil,zipfile
from collections import defaultdict,Counter
import numpy as np
import pandas as pd
from PIL import Image,ImageDraw,ImageFont,ImageFilter

OUT='/kaggle/working/qualitative_final_geometry'
os.makedirs(OUT,exist_ok=True)

def one(pattern,desc):
    xs=glob.glob(pattern,recursive=True)
    if not xs: raise FileNotFoundError(f'Missing {desc}. Pattern={pattern}')
    return sorted(xs,key=lambda x:(len(x),x))[0]

cc=[c for c in glob.glob('/kaggle/input/**/Geo-Bench-2D',recursive=True) if os.path.isdir(f'{c}/source_img')]
if not cc: raise FileNotFoundError('Geo-Bench-2D/source_img not found. Attach geobench2d-metrics-subset.')
CACHE=sorted(cc,key=len)[0]
COARSE=one('/kaggle/input/**/coarse_img/*/*/*.png','coarse_img').split('/coarse_img/')[0]+'/coarse_img'
GENBASE=one('/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup','full FreeFine baseline')
META=one('/kaggle/input/**/sample_metadata.csv','sample_metadata.csv')
ANNP=one('/kaggle/input/**/annotation_2d.json','annotation_2d.json')
ann=json.load(open(ANNP))

def png_count(p): return len(glob.glob(p+'/**/*.png',recursive=True)) if os.path.isdir(p) else 0
raw={}
for tag in ['EPSREC_PROMPT_ALL','MIDHF_EPSREC_PROMPT_ALL']:
    cands=glob.glob(f'/kaggle/input/**/finalB_guidance_move/variants/{tag}',recursive=True)
    if not cands: raise FileNotFoundError(f'Missing {tag}. Attach freefine-final-exhaustive-run-account-b-guidan')
    root=max(cands,key=png_count)
    assert png_count(root)>=200,(tag,png_count(root),root)
    raw[tag]=root

print('CACHE   =',CACHE)
print('COARSE  =',COARSE)
print('BASELINE=',GENBASE)
print('META    =',META)
print('ANN     =',ANNP)
print('RAW     =',raw)
print('✓ inputs resolved')


CACHE   = /kaggle/input/geobench2d-metrics-subset/geobench_metrics/Geo-Bench-2D
COARSE  = /kaggle/input/datasets/georgiostzamouranis/geobench2d-coarse-img/geobench_coarse/Geo-Bench-2D/coarse_img
BASELINE= /kaggle/input/datasets/georgiostzamouranis/freefine-geobench2d-bggen/gen_results_2d_final/gen_results_2d_backup
META    = /kaggle/input/datasets/georgiostzamouranis/freefine-sample-metadata/sample_metadata.csv
ANN     = /kaggle/input/geobench2d-metrics-subset/geobench_metrics/annotation_2d.json
RAW     = {'EPSREC_PROMPT_ALL': '/kaggle/input/notebooks/giorgostzam/freefine-final-exhaustive-run-account-b-guidan/finalB_guidance_move/variants/EPSREC_PROMPT_ALL', 'MIDHF_EPSREC_PROMPT_ALL': '/kaggle/input/notebooks/giorgostzam/freefine-final-exhaustive-run-account-b-guidan/finalB_guidance_move/variants/MIDHF_EPSREC_PROMPT_ALL'}
✓ inputs resolved


In [2]:
# Reconstruct exact historical balanced-200 subset
meta=[r for r in csv.DictReader(open(META))]
valid=[]
for r in meta:
    bp=f"{GENBASE}/{r['da_n']}/{r['ins_id']}/{r['case_id']}.png"
    if os.path.exists(bp): valid.append(r)

random.seed(42)
bycell=defaultdict(list)
for r in valid: bycell[(r['edit_type'],r['difficulty'])].append(r)
keys=sorted(bycell); per=200//len(keys); picked=[]
for k in keys:
    pool=bycell[k][:]; random.shuffle(pool); picked += pool[:per]
chosen={(r['da_n'],r['ins_id'],r['case_id']) for r in picked}
left=[r for r in valid if (r['da_n'],r['ins_id'],r['case_id']) not in chosen]
random.shuffle(left)
for r in left:
    if len(picked)>=200: break
    picked.append(r)
picked=picked[:200]
sha=hashlib.sha256('\n'.join(sorted(f"{r['da_n']}|{r['ins_id']}|{r['case_id']}" for r in picked)).encode()).hexdigest()
EXPECTED='3d7c0172cba1e35693a3f20b562004bcfd6fc43e187160c945a23224d40cbd3c'
assert sha==EXPECTED,(sha,EXPECTED)

for r in picked:
    ep=ann[str(r['da_n'])]['instances'][str(r['ins_id'])][str(r['case_id'])]['edit_param']
    dx,dy,dz,rx,ry,rz,sx,sy,sz=[float(x) for x in ep]
    r.update(dict(dx=dx,dy=dy,rz=rz,sx=sx,sy=sy,scale=math.sqrt(sx*sy)))
    r['affine_severe']=(r['edit_type']=='resize' and (r['scale']>=1.5 or r['scale']<=0.6))
    r['move_mag']=math.sqrt(dx*dx+dy*dy)
    r['rot_mag']=abs(rz)
print('✓ exact balanced-200',Counter(r['edit_type'] for r in picked))


✓ exact balanced-200 Counter({'move': 67, 'resize': 67, 'rotate': 66})


In [3]:
# Build exact RING4/RING8 post variants used in Step 1
try:
    import cv2
except Exception:
    cv2=None
RING4=os.path.join(OUT,'_ring4_move'); RING8=os.path.join(OUT,'_ring8_resize')
for p in [RING4,RING8]:
    if os.path.isdir(p): shutil.rmtree(p)
    os.makedirs(p,exist_ok=True)

def key(r): return str(r['da_n']),str(r['ins_id']),str(r['case_id'])
def target_mask_path(r):
    d,i,e=key(r); return f'{CACHE}/target_mask/{d}/{i}/{e}.png'
def source_path(r):
    d,_,_=key(r); return f'{CACHE}/source_img/{d}.png'
def coarse_path(r):
    d,i,e=key(r); return f'{COARSE}/{d}/{i}/{e}.png'
def baseline_path(r):
    d,i,e=key(r); return f'{GENBASE}/{d}/{i}/{e}.png'

def erode_mask(mask_bool,w):
    if cv2 is not None:
        return cv2.erode(mask_bool.astype(np.uint8),np.ones((2*w+1,2*w+1),np.uint8),iterations=1).astype(bool)
    im=Image.fromarray(mask_bool.astype(np.uint8)*255)
    return np.array(im.filter(ImageFilter.MinFilter(2*w+1)))>127

def ring_post(r,w):
    G=np.array(Image.open(baseline_path(r)).convert('RGB'))
    C=np.array(Image.open(coarse_path(r)).convert('RGB'))
    T=np.array(Image.open(target_mask_path(r)).convert('L'))>127
    interior=erode_mask(T,w)
    return Image.fromarray(np.where(interior[:,:,None],C,G).astype(np.uint8))

for r in picked:
    d,i,e=key(r)
    if r['edit_type']=='move':
        dst=f'{RING4}/{d}/{i}/{e}.png'; os.makedirs(os.path.dirname(dst),exist_ok=True); ring_post(r,4).save(dst)
    elif r['edit_type']=='resize':
        dst=f'{RING8}/{d}/{i}/{e}.png'; os.makedirs(os.path.dirname(dst),exist_ok=True); ring_post(r,8).save(dst)
print('✓ RING4/RING8 built')


✓ RING4/RING8 built


In [4]:
# Representative automatic selection
N_SEVERE_SHRINK=4; N_SEVERE_ENLARGE=4
N_NONSEVERE_SHRINK=3; N_NONSEVERE_ENLARGE=3
N_MOVE=6; N_ROTATE=6
MANUAL_CASES=[]  # e.g. [('455','0','3')]

resize=[r for r in picked if r['edit_type']=='resize']
sev_shrink=sorted([r for r in resize if r['affine_severe'] and r['scale']<1],key=lambda r:r['scale'])[:N_SEVERE_SHRINK]
sev_enlarge=sorted([r for r in resize if r['affine_severe'] and r['scale']>1],key=lambda r:-r['scale'])[:N_SEVERE_ENLARGE]
non_shrink=sorted([r for r in resize if not r['affine_severe'] and r['scale']<1],key=lambda r:r['scale'])[:N_NONSEVERE_SHRINK]
non_enlarge=sorted([r for r in resize if not r['affine_severe'] and r['scale']>1],key=lambda r:-r['scale'])[:N_NONSEVERE_ENLARGE]
moves=sorted([r for r in picked if r['edit_type']=='move'],key=lambda r:-r['move_mag'])[:N_MOVE]
rotates=sorted([r for r in picked if r['edit_type']=='rotate'],key=lambda r:-r['rot_mag'])[:N_ROTATE]
selected={'resize_severe':sev_shrink+sev_enlarge,'resize_nonsevere':non_shrink+non_enlarge,'move':moves,'rotate':rotates}
if MANUAL_CASES:
    byk={key(r):r for r in picked}
    selected['manual']=[byk[tuple(map(str,k))] for k in MANUAL_CASES if tuple(map(str,k)) in byk]

manifest=[]
for g,rs in selected.items():
    print('\n',g,len(rs))
    for rank,r in enumerate(rs,1):
        print(' ',key(r),'scale=',round(r['scale'],4),'move=',round(r['move_mag'],2),'rz=',round(r['rz'],2))
        manifest.append({'group':g,'rank':rank,'da_n':r['da_n'],'ins_id':r['ins_id'],'case_id':r['case_id'],'edit_type':r['edit_type'],'diagnostic_difficulty':r['difficulty'],'scale':r['scale'],'affine_severe':r['affine_severe'],'dx':r['dx'],'dy':r['dy'],'rz':r['rz']})
pd.DataFrame(manifest).to_csv(f'{OUT}/selected_cases_manifest.csv',index=False)
print('\n✓ selection manifest saved')



 resize_severe 8
  ('595', '0', '8') scale= 0.4197 move= 0.0 rz= 0.0
  ('377', '0', '10') scale= 0.4659 move= 0.0 rz= 0.0
  ('96', '0', '6') scale= 0.4904 move= 0.0 rz= 0.0
  ('272', '0', '9') scale= 0.4904 move= 0.0 rz= 0.0
  ('554', '0', '8') scale= 2.4398 move= 0.0 rz= 0.0
  ('376', '0', '7') scale= 1.9799 move= 0.0 rz= 0.0
  ('330', '0', '8') scale= 1.9356 move= 0.0 rz= 0.0
  ('46', '0', '4') scale= 1.8784 move= 0.0 rz= 0.0

 resize_nonsevere 6
  ('205', '0', '7') scale= 0.6005 move= 0.0 rz= 0.0
  ('204', '0', '5') scale= 0.6072 move= 0.0 rz= 0.0
  ('122', '0', '5') scale= 0.6081 move= 0.0 rz= 0.0
  ('512', '0', '6') scale= 1.4646 move= 0.0 rz= 0.0
  ('579', '0', '7') scale= 1.4101 move= 0.0 rz= 0.0
  ('453', '0', '7') scale= 1.4005 move= 0.0 rz= 0.0

 move 6
  ('463', '1', '1') scale= 1.0 move= 206.49 rz= 0.0
  ('420', '0', '2') scale= 1.0 move= 198.0 rz= 0.0
  ('72', '0', '2') scale= 1.0 move= 190.07 rz= 0.0
  ('342', '0', '2') scale= 1.0 move= 187.0 rz= 0.0
  ('195', '0', '2') 

In [5]:
# Render full-frame + target zoom grids
CELL=300; PAD=18; LABEL_H=44; ROW_META_H=34
BG=(245,245,245); BLACK=(20,20,20); BORDER=(180,180,180)
def font(size=18,bold=False):
    cands=[('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf' if bold else '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'),('/usr/share/fonts/truetype/liberation2/LiberationSans-Bold.ttf' if bold else '/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf')]
    for p in cands:
        if os.path.exists(p): return ImageFont.truetype(p,size)
    return ImageFont.load_default()
F18B=font(18,True); F14B=font(14,True)

def fit(im,size=CELL):
    im=im.convert('RGB'); w,h=im.size; sc=min(size/w,size/h); nw,nh=max(1,int(w*sc)),max(1,int(h*sc))
    rr=Image.Resampling.LANCZOS if hasattr(Image,'Resampling') else Image.LANCZOS
    im=im.resize((nw,nh),rr); c=Image.new('RGB',(size,size),(255,255,255)); c.paste(im,((size-nw)//2,(size-nh)//2)); return c

def target_bbox(r,pad_frac=0.18):
    m=np.array(Image.open(target_mask_path(r)).convert('L'))>127; ys,xs=np.where(m)
    if len(xs)==0: return (0,0,m.shape[1],m.shape[0])
    x0,x1=xs.min(),xs.max()+1; y0,y1=ys.min(),ys.max()+1; w=x1-x0; h=y1-y0; p=int(max(w,h)*pad_frac)+12
    return (max(0,x0-p),max(0,y0-p),min(m.shape[1],x1+p),min(m.shape[0],y1+p))

def method_paths(r,group):
    d,i,e=key(r); src=source_path(r); crs=coarse_path(r); bas=baseline_path(r)
    if group=='resize_severe': return [('Source',src),('Coarse affine',crs),('FreeFine',bas),('SGR-EPSREC',f"{raw['EPSREC_PROMPT_ALL']}/{d}/{i}/{e}.png"),('SGR-MIDHF+EPSREC',f"{raw['MIDHF_EPSREC_PROMPT_ALL']}/{d}/{i}/{e}.png")]
    if group=='resize_nonsevere': return [('Source',src),('Coarse affine',crs),('FreeFine',bas),('SGR-RING8',f'{RING8}/{d}/{i}/{e}.png')]
    if group=='move': return [('Source',src),('Coarse affine',crs),('FreeFine',bas),('SGR-RING4',f'{RING4}/{d}/{i}/{e}.png')]
    if group=='rotate': return [('Source',src),('Coarse affine',crs),('FreeFine / Final',bas)]
    if r['edit_type']=='resize' and r['affine_severe']: return method_paths(r,'resize_severe')
    if r['edit_type']=='resize': return method_paths(r,'resize_nonsevere')
    if r['edit_type']=='move': return method_paths(r,'move')
    return method_paths(r,'rotate')

def row_meta(r):
    d,i,e=key(r)
    if r['edit_type']=='resize': return f"{d}/{i}/{e} | resize scale={r['scale']:.3f} | affine {'severe' if r['affine_severe'] else 'non-severe'}"
    if r['edit_type']=='move': return f"{d}/{i}/{e} | move dx={r['dx']:.1f}, dy={r['dy']:.1f}, |t|={r['move_mag']:.1f}"
    return f"{d}/{i}/{e} | rotate rz={r['rz']:.1f}°"

def render_group(group,rows,zoom=False):
    if not rows: return None
    cols=max(len(method_paths(r,group)) for r in rows); W=PAD+cols*(CELL+PAD); H=PAD+LABEL_H+len(rows)*(ROW_META_H+CELL+PAD)
    can=Image.new('RGB',(W,H),BG); dr=ImageDraw.Draw(can)
    for j,h in enumerate([x[0] for x in method_paths(rows[0],group)]):
        x=PAD+j*(CELL+PAD); dr.text((x+4,PAD+8),h,font=F18B,fill=BLACK)
    y=PAD+LABEL_H
    for r in rows:
        dr.text((PAD,y+6),row_meta(r),font=F14B,fill=BLACK); y+=ROW_META_H; bbox=target_bbox(r) if zoom else None
        for j,(name,p) in enumerate(method_paths(r,group)):
            if not os.path.exists(p): raise FileNotFoundError(p)
            im=Image.open(p).convert('RGB'); im=im.crop(bbox) if bbox is not None else im; im=fit(im); x=PAD+j*(CELL+PAD); can.paste(im,(x,y)); dr.rectangle([x,y,x+CELL-1,y+CELL-1],outline=BORDER,width=1)
        y+=CELL+PAD
    suffix='zoom' if zoom else 'full'; p=f'{OUT}/{group}_{suffix}.png'; can.save(p); return p

created=[]
for g,rs in selected.items():
    for zoom in [False,True]:
        p=render_group(g,rs,zoom)
        if p: created.append(p)
print('Created figures:')
for p in created: print(' ',p)


Created figures:
  /kaggle/working/qualitative_final_geometry/resize_severe_full.png
  /kaggle/working/qualitative_final_geometry/resize_severe_zoom.png
  /kaggle/working/qualitative_final_geometry/resize_nonsevere_full.png
  /kaggle/working/qualitative_final_geometry/resize_nonsevere_zoom.png
  /kaggle/working/qualitative_final_geometry/move_full.png
  /kaggle/working/qualitative_final_geometry/move_zoom.png
  /kaggle/working/qualitative_final_geometry/rotate_full.png
  /kaggle/working/qualitative_final_geometry/rotate_zoom.png


In [6]:
from IPython.display import display,Markdown
for p in created:
    display(Markdown(f'### {os.path.basename(p)}'))
    display(Image.open(p))


### resize_severe_full.png

<PIL.PngImagePlugin.PngImageFile image mode=RGB size=1608x2878>

### resize_severe_zoom.png

<PIL.PngImagePlugin.PngImageFile image mode=RGB size=1608x2878>

### resize_nonsevere_full.png

...

<PIL.PngImagePlugin.PngImageFile image mode=RGB size=1290x2174>

### rotate_full.png

<PIL.PngImagePlugin.PngImageFile image mode=RGB size=972x2174>

### rotate_zoom.png

<PIL.PngImagePlugin.PngImageFile image mode=RGB size=972x2174>

In [7]:
ZIP='/kaggle/working/FreeFine_Final_Geometry_Qualitative_Comparison_v1.zip'
if os.path.exists(ZIP): os.remove(ZIP)
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f'{OUT}/*'):
        if os.path.isfile(p): z.write(p,arcname=os.path.basename(p))
print('✓ COMPLETE')
print('Figures:',len(created))
print('Manifest:',f'{OUT}/selected_cases_manifest.csv')
print('ZIP:',ZIP)


✓ COMPLETE
Figures: 8
Manifest: /kaggle/working/qualitative_final_geometry/selected_cases_manifest.csv
ZIP: /kaggle/working/FreeFine_Final_Geometry_Qualitative_Comparison_v1.zip
